# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via its Croissant schema URL:
<br>
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print a summary for context
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Dataset Description: {metadata.description}")
print(f"Authors: {[a for a in metadata.author]}")
print(f"Data collected: {metadata.dataCollection}")
print(f"Limitations: {metadata.dataLimitations}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant datasets structure their data as record sets, fields, and columns. All references use their `@id` values.

Let's list all record sets and their fields:

In [ ]:
# Get all record sets defined in metadata
record_sets = dataset.metadata.recordSet

if not record_sets:
    print("No record sets found in the metadata. Please check the schema or dataset definition.")
else:
    for rs in record_sets:
        print(f"Record set @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
        fields = rs.get('field', [])
        if fields:
            for fld in fields:
                print(f"  Field @id: {fld['@id']} | Name: {fld.get('name', 'N/A')} | DataType: {fld.get('dataType', 'N/A')}")
        else:
            print("  No fields listed.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview. If there are multiple record sets, load each one into its own DataFrame.
Let's extract all record sets (if any) as DataFrames.

In [ ]:
# List record set @ids
record_set_ids = []
for rs in dataset.metadata.recordSet:
    record_set_ids.append(rs['@id'])
print("Available record set @ids:", record_set_ids)

# Prepare dict of DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nRecord set {record_set_id} columns:", df.columns.tolist())
        print(df.head(5))
    else:
        print(f"No records found for {record_set_id}.")

# For subsequent steps, select the first populated record set
main_record_set_id = next(iter(dataframes), None)
if main_record_set_id:
    print(f"Main record set selected: {main_record_set_id}")
else:
    print("No record sets extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply simple filters, normalization, and group-by operations using the column `@id`s.

First, print available field (column) `@id`s for the main record set, then filter, normalize, and group as examples.

In [ ]:
# Check columns and prepare field @ids
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print("Columns in DataFrame:", df.columns.tolist())

    # Identify numeric and groupable fields by their @ids
    # For demonstration, select the first numeric-type column
    numeric_field_id = None
    group_field_id = None

    rs_obj = None
    for rs in dataset.metadata.recordSet:
        if rs['@id'] == main_record_set_id:
            rs_obj = rs
            break

    if rs_obj:
        for fld in rs_obj.get('field', []):
            if fld.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']:
                numeric_field_id = fld['@id']
            elif fld.get('dataType') == 'schema:Text':
                group_field_id = fld['@id']

    # If not found above, default to column names
    if not numeric_field_id:
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break
    if not group_field_id:
        for c in df.columns:
            if pd.api.types.is_string_dtype(df[c]):
                group_field_id = c
                break

    print(f"Numeric field @id selected: {numeric_field_id}")
    print(f"Group-by field @id selected: {group_field_id}")

    # Filtering: Choose arbitrary threshold (e.g., mean or fixed value)
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head(5))

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group-by
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"No numeric field found for EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields.

Below is a simple histogram and group-by bar plot. Ensure the selected fields are valid.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id:
    df = dataframes[main_record_set_id]
    if numeric_field_id and numeric_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

    if group_field_id and group_field_id in df.columns and numeric_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print("No record set available for visualization.")

## 6. Conclusion
This notebook demonstrated loading, exploring, processing, and visualizing a FAIR²-compliant Croissant dataset with `mlcroissant`. We:
- Loaded the metadata and records
- Identified record sets and used their `@id` values for extraction
- Performed filtering, normalization, grouping, and visualization using the field/column `@id`s
- Most importantly, ensured that all data operations referenced entities strictly by their `@id`, as required for reproducible FAIR² analysis.

Further analyses can extend these EDA steps to model building or advanced statistical testing.